In [1]:
import os
from osgeo import gdal
import isce
import isceobj

def convert_dem_to_isce(input_tif, output_dem):
    print(f"--- Converting {input_tif} to ISCE format ---")
    
    # 1. Convert GeoTIFF to raw ENVI binary (Int16 is standard for DEMs)
    ds = gdal.Open(input_tif)
    gdal.Translate(output_dem, ds, format='ENVI', outputType=gdal.GDT_Int16)
    
    # 2. Extract Geotransform to build the XML
    gt = ds.GetGeoTransform()
    width = ds.RasterXSize
    length = ds.RasterYSize
    ds = None # Close dataset

    # 3. Build the ISCE XML header
    demImage = isceobj.createDemImage()
    demImage.setFilename(output_dem)
    demImage.setWidth(width)
    demImage.setLength(length)
    demImage.setDataType('SHORT') 
    demImage.setAccessMode('read')
    
    # ISCE coordinates map to pixel centers, so we shift by half a pixel
    dictProp = {
        'Coordinate1': {'size': width, 'startingValue': gt[0] + (gt[1] / 2.0), 'delta': gt[1]},
        'Coordinate2': {'size': length, 'startingValue': gt[3] + (gt[5] / 2.0), 'delta': gt[5]}
    }
    demImage.init(dictProp)
    demImage.renderHdr()
    
    print(f"✅ Success! Created {output_dem} and {output_dem}.xml")

if __name__ == "__main__":
    convert_dem_to_isce("dem_10m_wgs84.tif", "dem_10m.wgs84")

--- Converting dem_10m_wgs84.tif to ISCE format ---


Warning 1: for band 1, nodata value has been clamped to -32768, the original value being out of range.


Writing geotrans to VRT for dem_10m.wgs84
✅ Success! Created dem_10m.wgs84 and dem_10m.wgs84.xml
